# Task 3: Local 7B Parameter LLM Quantization & Logit Extraction Pipeline

## Mathematics
- 4-bit Uniform Quantization: Scale $S = \frac{x_{max} - x_{min}}{2^b - 1}$, Zero Point $Z = \text{round}(\frac{-x_{min}}{S})$.
- Dynamic Shannon Entropy: $H(X) = - \sum P(x) \log_2 P(x)$.


In [1]:
import torch
import torch.nn.functional as F

# 4-bit uniform quantization and Shannon entropy logit analytical functions
def quantize_int4(tensor: torch.Tensor):
    qmin, qmax = 0, 15
    min_val, max_val = tensor.min(), tensor.max()
    scale = (max_val - min_val) / (qmax - qmin)
    zero_point = torch.round(-min_val / scale)
    q_tensor = torch.clamp(torch.round(tensor / scale) + zero_point, qmin, qmax).to(torch.uint8)
    return q_tensor, scale, zero_point

def compute_entropy(logits: torch.Tensor):
    probs = F.softmax(logits, dim=-1)
    return -torch.sum(probs * torch.log2(probs + 1e-9), dim=-1)


In [2]:
# Quantize simulated LLM layer weights and analyze token logit entropy across generation steps
weights = torch.randn(1024, 1024) # Simulated FP32 model layer
q_w, scale, zp = quantize_int4(weights)
reconstructed = scale * (q_w.to(torch.float32) - zp)

fp32_bytes = weights.element_size() * weights.nelement()
int4_bytes = q_w.element_size() * q_w.nelement()
mse_loss = torch.mean((weights - reconstructed)**2).item()

# Simulated decoding steps logit distributions
steps = ["The", "capital", "of"]
simulated_logits = torch.tensor([
    [10.5, 2.1, 0.5] + [0.0]*997,  # High confidence step
    [5.2, 4.8, 4.1] + [0.0]*997,   # High entropy uncertainty step
    [12.1, 1.0, 0.1] + [0.0]*997   # Very high confidence step
])

entropies = compute_entropy(simulated_logits)

print(f"FP32 Layer Size: {fp32_bytes / 1024:.2f} KB | INT4 Quantized Size: {int4_bytes / 1024:.2f} KB")
print(f"Quantization Compression Ratio: {100 * (1 - int4_bytes/fp32_bytes):.1f}% reduction")
print(f"Quantization Reconstruction Error (MSE): {mse_loss:.6f}
")
print("Dynamic Logit Entropy across Generation Steps:")
for tok, ent in zip(steps, entropies):
    print(f"  Token '{tok}': Entropy = {ent.item():.4f} bits")


Error during execution: unterminated f-string literal (detected at line 22) (<string>, line 22)
